# 06 — GITT diffusion coefficients

This notebook asks: *what solid-state diffusion coefficient D does a GITT experiment give, and how is a result
reviewed before it is exported?* It builds a small synthetic GITT record whose answer is known exactly, runs the
`gregorithm.gitt_ici` analysis and compares the results with the analytical values. It runs in seconds.

Units are SI inside the library: seconds, amperes, volts, m³, m² and m²/s. Positive current is charge.
The protocol (GITT or ICI) is always chosen by you; it is never guessed from pulse durations. Every result starts
as *provisional* and is exported only after you accept it.

In [ ]:
import math
import tempfile
from pathlib import Path

_session = tempfile.TemporaryDirectory(prefix="gregorithm-notebook-")
WORK = Path(_session.name)
print(f"Temporary result area: {WORK}")

## 1. A synthetic GITT record with a known answer

One rest of 60 s, then three charge pulses of 100 s at 1 mA, each followed by a 60 s rest:

- during a pulse the voltage rises as `E(t) = OCV + 0.005 V + 0.004 V/√s · √t` (an ideal square-root response);
- after each pulse the cell relaxes and the open-circuit voltage ends 10 mV higher (ΔEs = 0.01 V).

The file is written to the temporary folder; no external data are used.

In [ ]:
OCV_START = 3.5  # V
PULSE_S = 100  # τ, s
REST_S = 60  # s
CURRENT_A = 0.001  # 1 mA, charge
JUMP_V = 0.005  # instantaneous (ohmic) jump at the start of the pulse, V
SLOPE = 0.004  # dE/d√t during the pulse, V/√s
STEP_V = 0.01  # ΔEs between consecutive rests, V

lines = ["time_s,current_a,voltage_v"]
now, ocv = 0, OCV_START
for _ in range(REST_S):
    now += 1
    lines.append(f"{now},0,{ocv}")
for _ in range(3):
    for k in range(1, PULSE_S + 1):
        now += 1
        lines.append(f"{now},{CURRENT_A},{ocv + JUMP_V + SLOPE * math.sqrt(k):.15f}")
    ocv += STEP_V
    start = ocv + 0.003 * math.sqrt(REST_S)
    for k in range(1, REST_S + 1):
        now += 1
        lines.append(f"{now},0,{start - 0.003 * math.sqrt(k):.15f}")
SOURCE = WORK / "synthetic_gitt.csv"
SOURCE.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"{len(lines) - 1} samples written to {SOURCE.name}")

## 2. Import and elaborate

GITT/ICI always starts from an existing elaboration: the rest and constant-current segments found there are the
ones the analysis uses.

In [ ]:
from gregorithm import ElaborationRequest, ImportRequest, elaborate, import_source

imported = import_source(
    ImportRequest(SOURCE, WORK / "store", "gitt-cell", "import-job", "import-v1")
)
elaborated = elaborate(
    ElaborationRequest(
        imported.ref.path / "points" / "canonical.parquet",
        imported.report.content_hash,
        WORK / "store",
        "gitt-cell",
        "elaboration-job",
        "elaboration-v1",
    )
)
print(f"Cycles: {elaborated.cycles}; half-cycles: {elaborated.half_cycles}")

## 3. Choose the scope and the protocol

`load_canonical_scope` reads the verified elaboration for the chosen half-cycle (plus neighbouring context).
`events(Protocol.GITT)` associates each constant-current pulse with the rest before and after it.

In [ ]:
from gregorithm.gitt_ici import Protocol, build_event_data, load_canonical_scope

scope = load_canonical_scope(
    elaborated.ref.path, half_cycles=[1], elaboration_fingerprint=elaborated.fingerprint
)
events = scope.events(Protocol.GITT)
data = build_event_data(events, scope.segments, scope.samples)
for number, event in enumerate(events, start=1):
    print(
        f"Pulse {number}: rows {event.active_first_row}-{event.active_last_row}, "
        f"complete={event.complete}, constant current={event.constant_current}"
    )

## 4. Physical inputs, methods and analysis

GITT VA and GITT Δ need the active-phase volume V_act and the effective diffusion area A_eff, entered directly in
SI units and never inferred. Here V_act = 1·10⁻⁹ m³ (1 mm³) and A_eff = 1·10⁻⁴ m² (1 cm²). Without them the
intermediate quantities are still computed, but D is unavailable with a stated reason.

`plan_analysis` creates one task per pulse and method; `run_batch` computes them. Nothing is accepted yet.

In [ ]:
from gregorithm.gitt_ici import (
    AnalysisSettings,
    DiffusionMethod,
    MethodVariant,
    OcvSource,
    PhysicalInputs,
    ResultSnapshot,
    plan_analysis,
    run_batch,
)

VA = MethodVariant(DiffusionMethod.GITT_VA, OcvSource.REST_SAMPLE)
DELTA = MethodVariant(DiffusionMethod.GITT_DELTA, OcvSource.REST_SAMPLE)
settings = AnalysisSettings(geometry=PhysicalInputs(active_volume_m3=1e-9, effective_area_m2=1e-4))
plan = plan_analysis(data, variants=(VA, DELTA), settings=settings)
outcome = run_batch(plan.tasks, previous=ResultSnapshot())
snapshot = outcome.snapshot
print(f"{len(plan.tasks)} tasks computed")

## 5. Results against the analytical values

For this record the two methods have closed-form answers:

- GITT VA: `D = 4/π · (V_act/A_eff)² · [ΔEs / (τ · dE/d√t)]²`
- GITT Δ: `D = 4/(π τ) · (V_act/A_eff)² · (ΔEs / ΔEτ)²`, with ΔEτ = 0.005 + 0.004·√τ V

The quality colour is an advisory indicator of the square-root fit; it never blocks acceptance.

In [ ]:
ratio = 1e-9 / 1e-4
expected = {
    DiffusionMethod.GITT_VA: 4 / math.pi * ratio**2 * (STEP_V / (PULSE_S * SLOPE)) ** 2,
    DiffusionMethod.GITT_DELTA: 4
    / (math.pi * PULSE_S)
    * ratio**2
    * (STEP_V / (JUMP_V + SLOPE * math.sqrt(PULSE_S))) ** 2,
}
print(
    f"{'pulse':>5} {'method':>10} {'capacity [mAh]':>15} {'D [m²/s]':>12} "
    f"{'expected':>12} {'quality':>8} state"
)
for number, item in enumerate(data, start=1):
    for variant in (VA, DELTA):
        record = snapshot.current(variant.identity(item.event.event_id))
        method = variant.method
        print(
            f"{number:>5} {method.value:>10} {record.coordinates.capacity_start_ah * 1e3:>15.4f} "
            f"{record.result.diffusion_m2_s:>12.4e} {expected[method]:>12.4e} "
            f"{record.quality.overall.value:>8} {record.state.value}"
        )

## 6. Review

Acceptance is a scientific decision. `accept_all` accepts every available provisional result of one method and
configuration; single results can be accepted, rejected or revoked with `accept`, `reject` and `revoke`. Here we
accept GITT VA and leave GITT Δ provisional, so only GITT VA will be exported.

In [ ]:
snapshot = snapshot.accept_all(
    method=DiffusionMethod.GITT_VA, variant="default", ocv_source=OcvSource.REST_SAMPLE
)
for record in snapshot.records:
    print(f"{record.identity.method.value:>10} {record.identity.event_id[:8]} {record.state.value}")

## 7. Save and export

`publish_snapshot` saves the whole snapshot as an immutable, checksummed result generation;
`export_accepted_results` writes only accepted, current results with a finite D.

In [ ]:
import csv

from gregorithm import (
    DependencyRef,
    ExportFormat,
    SnapshotPublicationRequest,
    export_accepted_results,
    publish_snapshot,
)

published = publish_snapshot(
    snapshot,
    SnapshotPublicationRequest(
        WORK / "store",
        "gitt-cell",
        "gitt-job",
        "gitt-v1",
        {"protocol": "gitt", "methods": ["gitt_va", "gitt_delta"]},
        (DependencyRef("elaboration", elaborated.fingerprint),),
    ),
)
receipt = export_accepted_results(published.path, WORK / "gitt_accepted.csv", ExportFormat.CSV)
print(f"Exported rows: {receipt.rows}; columns: {len(receipt.columns)}")
with (WORK / "gitt_accepted.csv").open(newline="", encoding="utf-8") as stream:
    rows = list(csv.DictReader(stream))
print({key: rows[0][key] for key in list(rows[0])[:8]})

## 8. Figures

In [ ]:
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

points = pq.read_table(
    elaborated.ref.path / "points" / "elaborated.parquet", columns=["time_s", "voltage_v"]
).to_pydict()
fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 6))
top.plot(points["time_s"], points["voltage_v"])
top.set_xlabel("Time (s)")
top.set_ylabel("Voltage (V)")
top.set_title("Synthetic GITT record")
for variant, marker in ((VA, "o"), (DELTA, "s")):
    records = [snapshot.current(variant.identity(item.event.event_id)) for item in data]
    bottom.semilogy(
        [r.coordinates.capacity_start_ah * 1e3 for r in records],
        [r.result.diffusion_m2_s for r in records],
        marker=marker,
        linestyle="none",
        label=variant.method.value,
    )
bottom.set_xlabel("Capacity at pulse start (mAh)")
bottom.set_ylabel("D (m²/s)")
bottom.legend()
fig.tight_layout()
fig

Further steps not shown here: `measure_ir_drops` (pre- and post-pulse IR drops with the real samples used),
`measure_rest_ocv` (sampled and extrapolated OCV of every rest), manual or physically guided fit windows, the
ICI protocol and the analytical spherical fit (`delacourt_variant`, which needs the particle radius r_p).
See [the GITT/ICI method page](../../docs/methods/gitt-ici.md) for the exact rules.